In [1]:
import os 
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL"), temperature=0.9)

In [4]:
import getpass
import os 

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
vector_store = Chroma(
    collection_name = "banking-app",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

In [32]:
docs = []

In [33]:
from langchain_community.document_loaders import PyMuPDFLoader
for file_path in os.listdir("banks-data/"):
    print(f"Processing file: {file_path}")
    if file_path.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join("banks-data/", file_path))
        loaded_docs = loader.load()
        print(f"Loaded {len(loaded_docs)} documents from {file_path}")
        print(f"Loader : {loader}")
        docs.extend(loaded_docs)

Processing file: Asaan_Digital_Account.pdf
Loaded 2 documents from Asaan_Digital_Account.pdf
Loader : <langchain_community.document_loaders.pdf.PyMuPDFLoader object at 0x79133752f7a0>
Processing file: Account-Opening-Form-Mustaqeem-TC-01.01.2026-copy.pdf
Loaded 13 documents from Account-Opening-Form-Mustaqeem-TC-01.01.2026-copy.pdf
Loader : <langchain_community.document_loaders.pdf.PyMuPDFLoader object at 0x791337db2a20>
Processing file: User-Guide-Digitalization-of-FX-Cases-Customers.pdf
Loaded 5 documents from User-Guide-Digitalization-of-FX-Cases-Customers.pdf
Loader : <langchain_community.document_loaders.pdf.PyMuPDFLoader object at 0x79132f0ef230>
Processing file: Soneri-Debit-Card-IBFT-UPBS-Dispute-Form.pdf
Loaded 1 documents from Soneri-Debit-Card-IBFT-UPBS-Dispute-Form.pdf
Loader : <langchain_community.document_loaders.pdf.PyMuPDFLoader object at 0x791337db2b40>
Processing file: Silkbank_Amalgamation_Final_Website_FAQs_English.pdf
Loaded 6 documents from Silkbank_Amalgamation_F

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 5472 sub-documents.


In [36]:
batch_size = 1000  # safe size (well below limit)

document_ids = []

for i in range(0, len(all_splits), batch_size):
    batch = all_splits[i:i + batch_size]
    
    ids = vector_store.add_documents(documents=batch)
    document_ids.extend(ids)

    print(f"Inserted batch {i // batch_size + 1}")

Inserted batch 1
Inserted batch 2
Inserted batch 3
Inserted batch 4
Inserted batch 5
Inserted batch 6


In [37]:
print(document_ids[:3])


['54e4188b-04a6-4820-8f39-45e76673a5a1', '025069a2-0ce3-43c3-a4f1-dde37b353262', '3bfd3e24-bf1e-47fe-8369-9fc133b27b9f']


In [38]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [72]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
""" You are banking assistand that has access to a collection of banking documents. When you receive a question, you should first use the 'retrieve_context' tool to retrieve relevant information from the documents. Then, use the retrieved information to answer the question. Always use the retrieved information to support your answer, and cite the sources of the information you use. If you don't know the answer, say you don't know, but always try to provide some helpful information if possible.
You need to mention multiple banks everytime you answer.
"""
)
agent = create_agent(model, tools, system_prompt=prompt, checkpointer=InMemorySaver())

In [69]:
query = (
    "I am buying car for the first time help me with it which bank should i approach to make sure that i get the best deal also help me with interest rates mentioned?\n\n"
    # "Once you get the answer, look up common extensions of that method."
)


In [70]:
result = ""
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    result = event["messages"][-1].content

In [71]:
from pprint import pprint
pprint(result)

('When buying a car for the first time and looking for the best financing '
 "deal, it's essential to compare offers from multiple banks. Based on the "
 'information I retrieved, here are some options and details you should '
 'consider:\n'
 '\n'
 '1. **MCB Bank**:\n'
 '   - MCB Bank provides car loans to both self-employed and salaried '
 'customers with flexible and competitive markup rates. They also offer '
 'discounted rates for existing branch customers.\n'
 '   - They have a strong network of auto-dealers which can be beneficial when '
 'purchasing a vehicle.\n'
 '   - They waive the processing fee for women borrowers and offer a 50% '
 'waiver for specially abled individuals.\n'
 '\n'
 '2. **Standard Chartered Bank (SOC)**:\n'
 '   - They have specific auto finance products with competitive interest '
 'rates.\n'
 '   - Processing fees range from Rs. 8,000 for amounts up to Rs. 1 million to '
 'Rs. 12,000 for amounts over Rs. 2 million, with waivers for women and '
 'disabled 

In [7]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="./banks-data/HistPR-2026.pdf",
    strategy="hi_res"   # better for text-based PDFs
)

ModuleNotFoundError: No module named 'unstructured_inference'

In [12]:
import os
import json
import pdfplumber
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
# from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver


In [13]:
# !pip install pdfplumber

In [14]:

# ── 1. Embeddings + Vector Store ───────────────────────────────────────────
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)


In [15]:

vector_store = Chroma(
    collection_name="banking-app",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)


In [16]:

# ── 2. PDF Extraction (text + structured tables) ───────────────────────────
def table_to_text(headers: list, rows: list[dict]) -> str:
    lines = ["Table columns: " + ", ".join(headers)]
    for row in rows:
        lines.append(" | ".join(f"{k}: {v}" for k, v in row.items() if v))
    return "\n".join(lines)



In [19]:

def extract_docs_from_pdf(pdf_path: str) -> list[Document]:
    docs = []
    filename = os.path.basename(pdf_path)

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables      = page.find_tables()
            table_bboxes = [t.bbox for t in tables]

            # ── Text (excluding table regions) ──
            # NEW
            text_page = page
            for bbox in table_bboxes:
                clamped = clamp_bbox(bbox, page.bbox)
                try:
                    text_page = text_page.outside_bbox(clamped)
                except ValueError:
                    pass  # skip if still out of bounds after clamping

            raw_text = text_page.extract_text()
            if raw_text and raw_text.strip():
                docs.append(Document(
                    page_content=raw_text.strip(),
                    metadata={
                        "source":   filename,
                        "page":     page_num,
                        "type":     "text"
                    }
                ))

            # ── Tables ──
            for tbl_idx, table in enumerate(tables):
                rows = table.extract()
                if not rows or len(rows) < 2:
                    continue

                raw_headers = rows[0]
                headers = [
                    (str(h).strip() if h and str(h).strip() else f"col_{i}")
                    for i, h in enumerate(raw_headers)
                ]
                structured_rows = []
                for row in rows[1:]:
                    record = {
                        headers[i] if i < len(headers) else f"col_{i}": 
                        (str(cell).strip() if cell else "")
                        for i, cell in enumerate(row)
                    }
                    structured_rows.append(record)

                docs.append(Document(
                    page_content=table_to_text(headers, structured_rows),
                    metadata={
                        "source":      filename,
                        "page":        page_num,
                        "type":        "table",
                        "table_index": tbl_idx,
                        "headers":     json.dumps(headers),
                        "raw_json":    json.dumps(structured_rows)
                    }
                ))
    return docs

def clamp_bbox(bbox, page_bbox):
    x0, top, x1, bottom = bbox
    px0, ptop, px1, pbottom = page_bbox
    return (
        max(x0, px0),
        max(top, ptop),
        min(x1, px1),
        min(bottom, pbottom)
    )



In [20]:

# ── 3. Load all PDFs ───────────────────────────────────────────────────────
all_docs = []
for filename in os.listdir("banks-data/"):
    if filename.endswith(".pdf"):
        path = os.path.join("banks-data/", filename)
        print(f"Processing: {filename}")
        extracted = extract_docs_from_pdf(path)
        print(f"  → {len(extracted)} chunks (text + tables)")
        all_docs.extend(extracted)

print(f"\nTotal chunks: {len(all_docs)}")

# ── 4. Split text chunks only (tables stay intact) ────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter


Processing: Asaan_Digital_Account.pdf
  → 6 chunks (text + tables)
Processing: Account-Opening-Form-Mustaqeem-TC-01.01.2026-copy.pdf
  → 16 chunks (text + tables)
Processing: User-Guide-Digitalization-of-FX-Cases-Customers.pdf
  → 6 chunks (text + tables)
Processing: Soneri-Debit-Card-IBFT-UPBS-Dispute-Form.pdf
  → 0 chunks (text + tables)
Processing: Silkbank_Amalgamation_Final_Website_FAQs_English.pdf
  → 6 chunks (text + tables)
Processing: KFS-MPA-EN.pdf
  → 4 chunks (text + tables)
Processing: KFS-MDA-EN.pdf
  → 3 chunks (text + tables)
Processing: 1st-Quarterly-March-2018.pdf
  → 44 chunks (text + tables)
Processing: KFS-MARA-EN.pdf
  → 4 chunks (text + tables)
Processing: Ann_2022.pdf
  → 463 chunks (text + tables)
Processing: SOC-ENG-Jan-Jun-2026.pdf
  → 31 chunks (text + tables)
Processing: Guidelines-for-Business-Conduct-for-Banks.pdf
  → 32 chunks (text + tables)
Processing: SOC-01-Jan-2025.pdf
  → 159 chunks (text + tables)
Processing: Service-Standards-Booklet.pdf
  → 12 c

In [22]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)


In [23]:

final_chunks = []
for doc in all_docs:
    if doc.metadata.get("type") == "table":
        final_chunks.append(doc)          # keep tables whole
    else:
        final_chunks.extend(text_splitter.split_documents([doc]))

print(f"Final chunks after splitting: {len(final_chunks)}")


Final chunks after splitting: 6307


In [26]:
from pprint import pprint
pprint(final_chunks[:3])

[Document(metadata={'source': 'Asaan_Digital_Account.pdf', 'page': 1, 'type': 'text', 'start_index': 0}, page_content='*Above charges are in PKR (PKR equivalent to be charged in case of FCY accounts)'),
 Document(metadata={'source': 'Asaan_Digital_Account.pdf', 'page': 1, 'type': 'table', 'table_index': 0, 'headers': '["Retail", "Deposit Product - Key Fact Sheet", "col_2", "col_3"]', 'raw_json': '[{"Retail": "Asaan Digital\\nSelect Product Here:\\nCurrent Account\\nPKR (701)", "Deposit Product - Key Fact Sheet": "", "col_2": "", "col_3": "IMPORTANT: Read this document carefully if you are considering opening a new account. You may also use this\\ndocument to compare different accounts offered by other banks. You have the right to receive KFS from other\\nbanks for comparison."}, {"Retail": "Product Type", "Deposit Product - Key Fact Sheet": "", "col_2": "Islamic Current Account", "col_3": "This information is accurate as of the date below. Products/Services and/or its fees may change f

In [27]:

# ── 5. Ingest into ChromaDB in batches ────────────────────────────────────
BATCH_SIZE = 500
document_ids = []

for i in range(0, len(final_chunks), BATCH_SIZE):
    batch = final_chunks[i:i + BATCH_SIZE]
    ids = vector_store.add_documents(documents=batch)
    document_ids.extend(ids)
    print(f"Inserted batch {i // BATCH_SIZE + 1} ({len(batch)} chunks)")

print(f"\nTotal inserted: {len(document_ids)} chunks")


Inserted batch 1 (500 chunks)
Inserted batch 2 (500 chunks)
Inserted batch 3 (500 chunks)
Inserted batch 4 (500 chunks)
Inserted batch 5 (500 chunks)
Inserted batch 6 (500 chunks)
Inserted batch 7 (500 chunks)
Inserted batch 8 (500 chunks)
Inserted batch 9 (500 chunks)
Inserted batch 10 (500 chunks)
Inserted batch 11 (500 chunks)
Inserted batch 12 (500 chunks)
Inserted batch 13 (307 chunks)

Total inserted: 6307 chunks


In [28]:

# ── 6. RAG Tool ───────────────────────────────────────────────────────────
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve relevant banking information including tables and text."""
    retrieved_docs = vector_store.similarity_search(query, k=5)

    parts = []
    for doc in retrieved_docs:
        meta = doc.metadata
        chunk_type = meta.get("type", "text")

        if chunk_type == "table":
            # Return structured table data for the LLM
            try:
                table_data = json.loads(meta.get("raw_json", "[]"))
                headers    = json.loads(meta.get("headers", "[]"))
                table_str  = f"[TABLE] Source: {meta['source']} | Page {meta['page']}\n"
                table_str += f"Columns: {', '.join(headers)}\n"
                table_str += doc.page_content
            except Exception:
                table_str = doc.page_content
            parts.append(table_str)
        else:
            parts.append(
                f"[TEXT] Source: {meta.get('source')} | Page {meta.get('page')}\n"
                f"{doc.page_content}"
            )

    serialized = "\n\n---\n\n".join(parts)
    return serialized, retrieved_docs


# ── 7. LLM + Agent ────────────────────────────────────────────────────────
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)


In [29]:

tools = [retrieve_context]

system_prompt = """You are a banking assistant with access to a collection of banking documents.

When you receive a question:
1. Always use the 'retrieve_context' tool first to find relevant information.
2. Use retrieved data — especially tables for interest rates, fees, and terms — to support your answer.
3. Always compare MULTIPLE banks in your response.
4. Cite the source document and page number for every claim.
5. If the answer isn't in the documents, say so honestly.
"""

agent = create_react_agent(
    model,
    tools,
    prompt=system_prompt,
    checkpointer=MemorySaver()
)


/tmp/ipykernel_5793/782848668.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [30]:

# ── 8. Run a query ────────────────────────────────────────────────────────
config = {"configurable": {"thread_id": "session-1"}}  # enables memory


In [31]:

response = agent.invoke(
    {"messages": [{"role": "user", "content": 
        "I am buying a car for the first time. Which bank should I approach "
        "for the best deal? Help me with the interest rates mentioned."
    }]},
    config=config
)

print("\n=== Agent Response ===")
print(response["messages"][-1].content)


=== Agent Response ===
When considering a car loan, it's essential to compare the interest rates, processing fees, and other charges from different banks. Here’s a summary of the relevant details from various banks:

### 1. **Bank A (SOC)**
- **Processing Fee**: 
  - Up to Rs. 1 Million: Rs. 8,000
  - Rs. 1 Million to Rs. 2 Million: Rs. 10,000
  - Above Rs. 2 Million: Rs. 12,000
  - **Waivers**: 100% for women borrowers, 50% for specially abled persons (Page 13).
  
- **Premature Termination Charges**:
  - 1st Year: 8% of Principal Outstanding
  - 2nd Year: 6%
  - 3rd Year and onwards: 5% (Page 14).

### 2. **Bank B (SOBC)**
- **Processing Fee**: 
  - Minimum Rs. 2,000 or 0.20% of the finance amount, whichever is higher (Page 29).
  
- **Prepayment Fee**:
  - 1st Year: Up to 8% of the principal amount prepaid
  - 2nd Year: Up to 6%
  - 3rd Year and onwards: Up to 3.5% (Page 29).

### 3. **Bank C (UBL)**
- **Processing Fee**: 
  - New/Used/Local Car: Up to Rs. 12,000 (to be received af

In [2]:
import os
import json
import pdfplumber
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# ── 1. Embeddings + Vector Store ───────────────────────────────────────────
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

vector_store = Chroma(
    collection_name="banking-app",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)


In [3]:

# ── 2. Helper Functions (MUST be defined before use) ──────────────────────
def clamp_bbox(bbox, page_bbox):
    x0, top, x1, bottom = bbox
    px0, ptop, px1, pbottom = page_bbox
    return (
        max(x0, px0),
        max(top, ptop),
        min(x1, px1),
        min(bottom, pbottom)
    )


In [4]:

def table_to_text(headers: list, rows: list[dict]) -> str:
    lines = ["Table columns: " + ", ".join(headers)]
    for row in rows:
        lines.append(" | ".join(f"{k}: {v}" for k, v in row.items() if v))
    return "\n".join(lines)


In [5]:

def get_bank_name(filename: str) -> str:
    name_map = {
        "SOC":  "State Bank of Pakistan (SBP)",
        "SOBC": "Bank of China (SOBC)",
        "UBL":  "United Bank Limited (UBL)",
        "HBL":  "Habib Bank Limited (HBL)",
        "MCB":  "MCB Bank",
        "ABL":  "Allied Bank Limited (ABL)",
        "NBP":  "National Bank of Pakistan (NBP)",
    }
    filename_upper = filename.upper()
    for key, full_name in name_map.items():
        if key in filename_upper:
            return full_name
    return filename.replace(".pdf", "")  # fallback


In [6]:

def extract_docs_from_pdf(pdf_path: str) -> list[Document]:
    docs = []
    filename = os.path.basename(pdf_path)
    bank_name = get_bank_name(filename)

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables       = page.find_tables()
            table_bboxes = [t.bbox for t in tables]

            # ── Text (excluding table regions) ──
            text_page = page
            for bbox in table_bboxes:
                clamped = clamp_bbox(bbox, page.bbox)
                try:
                    text_page = text_page.outside_bbox(clamped)
                except ValueError:
                    pass

            raw_text = text_page.extract_text()
            if raw_text and raw_text.strip():
                docs.append(Document(
                    page_content=raw_text.strip(),
                    metadata={
                        "source":    filename,
                        "bank_name": bank_name,
                        "page":      page_num,
                        "type":      "text"
                    }
                ))

            # ── Tables ──
            for tbl_idx, table in enumerate(tables):
                rows = table.extract()
                if not rows or len(rows) < 2:
                    continue

                raw_headers = rows[0]
                headers = [
                    (str(h).strip() if h and str(h).strip() else f"col_{i}")
                    for i, h in enumerate(raw_headers)
                ]
                structured_rows = []
                for row in rows[1:]:
                    record = {
                        headers[i] if i < len(headers) else f"col_{i}":
                        (str(cell).strip() if cell else "")
                        for i, cell in enumerate(row)
                    }
                    structured_rows.append(record)

                docs.append(Document(
                    page_content=table_to_text(headers, structured_rows),
                    metadata={
                        "source":      filename,
                        "bank_name":   bank_name,
                        "page":        page_num,
                        "type":        "table",
                        "table_index": tbl_idx,
                        "headers":     json.dumps(headers),
                        "raw_json":    json.dumps(structured_rows)
                    }
                ))
    return docs


In [7]:

# ── 3. Load all PDFs ───────────────────────────────────────────────────────
all_docs = []
for filename in os.listdir("banks-data/"):
    if filename.endswith(".pdf"):
        path = os.path.join("banks-data/", filename)
        print(f"Processing: {filename}")
        extracted = extract_docs_from_pdf(path)
        print(f"  → {len(extracted)} chunks (text + tables)")
        all_docs.extend(extracted)

print(f"\nTotal chunks: {len(all_docs)}")


Processing: Asaan_Digital_Account.pdf
  → 6 chunks (text + tables)
Processing: Account-Opening-Form-Mustaqeem-TC-01.01.2026-copy.pdf
  → 16 chunks (text + tables)
Processing: User-Guide-Digitalization-of-FX-Cases-Customers.pdf
  → 6 chunks (text + tables)
Processing: Soneri-Debit-Card-IBFT-UPBS-Dispute-Form.pdf
  → 0 chunks (text + tables)
Processing: Silkbank_Amalgamation_Final_Website_FAQs_English.pdf
  → 6 chunks (text + tables)
Processing: KFS-MPA-EN.pdf
  → 4 chunks (text + tables)
Processing: KFS-MDA-EN.pdf
  → 3 chunks (text + tables)
Processing: 1st-Quarterly-March-2018.pdf
  → 44 chunks (text + tables)
Processing: KFS-MARA-EN.pdf
  → 4 chunks (text + tables)
Processing: Ann_2022.pdf
  → 463 chunks (text + tables)
Processing: SOC-ENG-Jan-Jun-2026.pdf
  → 31 chunks (text + tables)
Processing: Guidelines-for-Business-Conduct-for-Banks.pdf
  → 32 chunks (text + tables)
Processing: SOC-01-Jan-2025.pdf
  → 159 chunks (text + tables)
Processing: Service-Standards-Booklet.pdf
  → 12 c

In [8]:

# ── 4. Split text chunks only (tables stay intact) ────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

final_chunks = []
for doc in all_docs:
    if doc.metadata.get("type") == "table":
        final_chunks.append(doc)
    else:
        final_chunks.extend(text_splitter.split_documents([doc]))

print(f"Final chunks after splitting: {len(final_chunks)}")


Final chunks after splitting: 6307


In [9]:

# ── 5. Ingest into ChromaDB in batches ────────────────────────────────────
BATCH_SIZE = 500
document_ids = []

for i in range(0, len(final_chunks), BATCH_SIZE):
    batch = final_chunks[i:i + BATCH_SIZE]
    ids = vector_store.add_documents(documents=batch)
    document_ids.extend(ids)
    print(f"Inserted batch {i // BATCH_SIZE + 1} ({len(batch)} chunks)")

print(f"Total inserted: {len(document_ids)} chunks")


Inserted batch 1 (500 chunks)
Inserted batch 2 (500 chunks)
Inserted batch 3 (500 chunks)
Inserted batch 4 (500 chunks)
Inserted batch 5 (500 chunks)
Inserted batch 6 (500 chunks)
Inserted batch 7 (500 chunks)
Inserted batch 8 (500 chunks)
Inserted batch 9 (500 chunks)
Inserted batch 10 (500 chunks)
Inserted batch 11 (500 chunks)
Inserted batch 12 (500 chunks)
Inserted batch 13 (307 chunks)
Total inserted: 6307 chunks


In [22]:

# ── 6. RAG Tool ───────────────────────────────────────────────────────────
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve relevant banking information including tables and text."""
    retrieved_docs = vector_store.similarity_search(query, k=5)

    parts = []
    for doc in retrieved_docs:
        meta = doc.metadata
        bank_name = meta.get("bank_name", meta.get("source", "Unknown Bank"))

        if meta.get("type") == "table":
            try:
                headers = json.loads(meta.get("headers", "[]"))
                table_str  = f"[SOURCE]\nBank: {bank_name}\nFile: {meta['source']}\nPage: {meta['page']}\n\n"
                table_str += f"Columns: {', '.join(headers)}\n"
                table_str += doc.page_content
            except Exception:
                table_str = doc.page_content
            parts.append(table_str)
        else:
            parts.append(
                f"[TEXT] Bank: {bank_name} | Page {meta.get('page')}\n"
                f"{doc.page_content}"
            )

    return "\n\n---\n\n".join(parts), retrieved_docs


In [ ]:

# ── 7. LLM + Agent ────────────────────────────────────────────────────────
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

tools = [retrieve_context]

system_prompt = """You are a banking assistant with access to Pakistani banking documents.
You are a banking assistant with access to Pakistani banking documents.

STRICT RULES (MANDATORY):
- Every numerical value MUST include citation.
- Citation format MUST be exactly:
  (Bank: <bank_name>, Page: <page>, Source: <filename>)


- NEVER write any number without citation.
- If multiple banks are compared, EACH row must include citation.

- DO NOT summarize without attribution.
If exact information about car loans or auto finance is NOT found:
- DO NOT guess
- DO NOT generalize across banks
- Explicitly say:
  "This document does not contain car loan details"
When answering:
1. Always use the 'retrieve_context' tool first.
2. ALWAYS refer to banks by their REAL name from the metadata (bank_name field), never as "Bank A" or "Bank B".
3. Compare multiple banks in every response.
4. Cite the source document, bank name, and page number for every claim.
5. Use tables from retrieved data for interest rates and fees.

Use the following mapping to identify banks from filenames:
        "SOC":  "State Bank of Pakistan (SBP)",
        "SOBC": "Bank of China (SOBC)",
        "UBL":  "United Bank Limited (UBL)",
        "HBL":  "Habib Bank Limited (HBL)",
        "MCB":  "MCB Bank",
        "ABL":  "Allied Bank Limited (ABL)",
        "NBP":  "National Bank of Pakistan (NBP)", 
"""

agent = create_react_agent(
    model,
    tools,
    prompt=system_prompt,
    checkpointer=MemorySaver()
)


/tmp/ipykernel_5117/2908409260.py:40: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [24]:

# ── 8. Run a query ────────────────────────────────────────────────────────
config = {"configurable": {"thread_id": "session-1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content":
        "I am buying a car for the first time. Which bank should I approach "
        "for the best deal? Help me with the interest rates mentioned."
    }]},
    config=config
)

print("\n=== Agent Response ===")
print(response["messages"][-1].content)


=== Agent Response ===
When considering a car loan, it's essential to compare the interest rates and fees from different banks. Below is a comparison of the processing fees and premature termination charges for car loans from various banks:

| Bank Name                                   | Processing Fee (up to Rs. 1 Million) | Premature Termination Charges (1st Year) | Premature Termination Charges (2nd Year) | Premature Termination Charges (3rd Year and onwards) |
|---------------------------------------------|--------------------------------------|------------------------------------------|------------------------------------------|------------------------------------------------------|
| State Bank of Pakistan (SBP)               | Rs. 8,000/-                          | 8.0% of Principal Outstanding            | 6.0% of Principal Outstanding            | 5.0% of Principal Outstanding                          |
| Bank of China (SOBC)                       | Rs. 8,000/-              

In [20]:

# ── 8. Run a query ────────────────────────────────────────────────────────
config = {"configurable": {"thread_id": "session-1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content":
        """I want to buy a car with latest specs and i want to have the best deal possible which bank should i approach? also help me with the interest rates mentioned in the documents. enlist 2 banks with their interest rates and other details mentioned in the documents."""
    }]},
    config=config
)

print("\n=== Agent Response ===")
print(response["messages"][-1].content)


=== Agent Response ===
To help you find the best deal for a car loan with the latest specifications, I've compared two banks: **State Bank of Pakistan (SBP)** and **Bank of China (SOBC)**. Below are the details regarding their interest rates and other relevant fees.

### Car Loan Comparison

| **Bank**                          | **Processing Fee**                          | **1st Year Prepayment Fee** | **2nd Year Prepayment Fee** | **3rd Year Prepayment Fee** | **Special Waivers** |
|-----------------------------------|---------------------------------------------|------------------------------|------------------------------|------------------------------|----------------------|
| **State Bank of Pakistan (SBP)**  | Up to Rs. 12,000 (varies by amount)       | 8.0% of Principal Outstanding | 6.0% of Principal Outstanding | 5.0% of Principal Outstanding | 100% for women borrowers; 50% for specially abled persons (Page 14) |
| **Bank of China (SOBC)**          | Minimum Rs. 2,000 or 0.2